# Network Metrics

Reads the `#netvis` edge file for this survey, builds a NetworkX graph, and computes node-level metrics: `PageRank#number`, `Betweenness#number`, `Degree#number`, and `Community#number` (Louvain). Joins them back to survey rows by node identifier.

**Requires:** survey has network visualization data (`has_netvis` capability).

In [ ]:
# -- Colab bootstrap (no-op on Binder / JupyterHub) ------------------------
import os, sys
if "google.colab" in sys.modules:
    import subprocess
    if os.path.isdir("/tmp/suave-nb/.git"):
        subprocess.run(["git","-C","/tmp/suave-nb","fetch","--depth=1","origin","main"],
                       capture_output=True)
        subprocess.run(["git","-C","/tmp/suave-nb","reset","--hard","origin/main"],
                       capture_output=True)
    else:
        _r = subprocess.run(
            ["git","clone","--depth=1",
             "https://github.com/izaslavsky/suave-notebooks.git","/tmp/suave-nb"],
            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# ── Colab only: mount Google Drive to load session credentials ─────────────
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

if not su.ENV.colab:
    # Binder / JupyterHub: parameters load automatically
    _p = su.load_params()
    if _p is None:
        from IPython.display import HTML as _HTML
        display(_HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE parameters found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open this notebook via SuAVEDispatch from your survey in SuAVE.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
else:
    # Colab: try Drive/cache silently; credentials cell below handles fallback
    _p = su.load_params(_silent=True)


In [ ]:
# -- Colab: verify session loaded from Drive/cache --------------------------
from IPython.display import HTML
if su.ENV.colab:
    if _p is None:
        display(HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE session found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open SuAVEDispatch from the correct survey in SuAVE, then re-open this notebook.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
    display(HTML(
        '<div style="font-size:12px;padding:8px 12px;border-radius:4px;margin:3px 0;'
        'background:#e8f5e9;border:1px solid #81c784">'
        'Session loaded &mdash; survey <b>' + _p.get('survey', '?') + '</b>'
        ', user <b>' + _p.get('user', '?') + '</b>.'
        '<br><span style="color:#666;font-size:11px">'
        'Wrong survey? Go back to SuAVE, open the correct survey, and click Jupyter again.'
        '</span></div>'))
else:
    su._skipped('Colab only')


In [ ]:
# ── Variables used throughout this notebook ────────────────────────────────
if _p is None:
    from IPython.display import HTML as _HTML
    display(_HTML(
        '<div style="background:#dc3545;color:white;padding:14px 16px;'
        'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
        '&#9888; Parameters not loaded.'
        '<br><span style="font-size:12px;font-weight:normal">'
        'Navigate to your survey in SuAVE and relaunch this notebook to load parameters.'
        '</span></div>'))
    class _Stop(Exception):
        def _render_traceback_(self): return []
    raise _Stop()
user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'

localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images

if not _caps.get('has_netvis'):
    raise RuntimeError('This survey does not have network visualization data.')

absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)

In [ ]:
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint

import ipywidgets as widgets
import pandas as pd
import numpy as np
import networkx as nx
import io

## 1. Load survey data and fetch network edge file

In [ ]:
df = panellibs.extract_data(absolutePath + csv_file)
print(f"Loaded {len(df)} rows")

# Fetch netvis file list from /getSurvey
_origin = _urlparse(survey_url)
_host   = f"{_origin.scheme}://{_origin.netloc}"
_rec    = _req.get(
    f"{_host}/getSurvey",
    params={'name': _p.get('survey', ''), 'user': user},
    timeout=10
).json()

netvis_files = _rec.get('netvis', [])
if not netvis_files:
    raise RuntimeError('No netvis files found in survey record.')

netvis_url = f"{_host}/{netvis_files[0]}"
print(f"Network file: {netvis_url}")

resp = _req.get(netvis_url, timeout=30)
resp.raise_for_status()
edge_df = pd.read_csv(io.StringIO(resp.text))
print(f"Edge file columns: {edge_df.columns.tolist()}")
display(edge_df.head())

## 2. Configure graph construction

In [ ]:
edge_cols = edge_df.columns.tolist()

src_col = widgets.Dropdown(
    options=edge_cols, value=edge_cols[0],
    description='Source column:', layout=widgets.Layout(width='60%')
)
tgt_col = widgets.Dropdown(
    options=edge_cols, value=edge_cols[1] if len(edge_cols) > 1 else edge_cols[0],
    description='Target column:', layout=widgets.Layout(width='60%')
)
directed = widgets.Checkbox(value=False, description='Directed graph')

# Column in survey CSV that matches node identifiers
survey_cols = df.columns.tolist()
node_col = widgets.Dropdown(
    options=survey_cols, description='Survey node column:',
    layout=widgets.Layout(width='60%')
)
display(src_col, tgt_col, directed, node_col)

## 3. Build graph and compute metrics

In [ ]:
# Build graph
G = nx.DiGraph() if directed.value else nx.Graph()
for _, row in edge_df.iterrows():
    G.add_edge(str(row[src_col.value]), str(row[tgt_col.value]))

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Compute metrics
printmd('**Computing PageRank...**')
pagerank = nx.pagerank(G, alpha=0.85)

printmd('**Computing betweenness centrality...**')
betweenness = nx.betweenness_centrality(G)

printmd('**Computing degree...**')
degree = dict(G.degree())

printmd('**Computing Louvain communities...**')
try:
    from community import best_partition
except ImportError:
    import subprocess, sys as _sys
    subprocess.check_call([_sys.executable, '-m', 'pip', 'install', '-q', 'python-louvain'])
    from community import best_partition

Gu = G.to_undirected() if directed.value else G
partition = best_partition(Gu)

# Build metric DataFrame keyed by node
node_metrics = pd.DataFrame({
    'node':           list(pagerank.keys()),
    'PageRank#number':    [round(v, 6) for v in pagerank.values()],
    'Betweenness#number': [round(betweenness.get(n, 0), 6) for n in pagerank.keys()],
    'Degree#number':      [degree.get(n, 0) for n in pagerank.keys()],
    'Community#number':   [partition.get(n, -1) for n in pagerank.keys()],
})

printmd('**Metric summary (top 10 by PageRank):**')
display(node_metrics.sort_values('PageRank#number', ascending=False).head(10))

## 4. Join metrics to survey rows

In [ ]:
nc = node_col.value
df[nc] = df[nc].astype(str)
df = df.merge(
    node_metrics.rename(columns={'node': nc}),
    on=nc, how='left'
)

matched = df['PageRank#number'].notna().sum()
printmd(f"**Matched {matched}/{len(df)} survey rows to network nodes.**")
display(df[[nc, 'PageRank#number', 'Betweenness#number', 'Degree#number', 'Community#number']].head(10))

## 5. Save and publish to SuAVE

In [ ]:
new_file = suaveint.save_csv_file(df, absolutePath, csv_file)

In [ ]:
#Input survey name
survey_name = input('Enter survey name: ')
printmd('**Survey Name:** ' + survey_name)
